In [0]:
%run ../00_utils

In [0]:
from pyspark.sql.window import Window

def build_dim_date(start="2015-01-01", end="2027-12-31"):
    first_policy = spark.table("workspace.gold.ref_policy_rate").agg(F.min("effective_date")).collect()[0][0]
    days = spark.sql(f"SELECT explode(sequence(DATE'{first_policy}', DATE'{end}', INTERVAL 1 DAY)) AS full_date")

    pol = spark.table("workspace.gold.ref_policy_rate").select(
        F.col("effective_date").alias("full_date"), "policy_rate_pct", "rate_change_bps")
    w = Window.orderBy("full_date").rowsBetween(Window.unboundedPreceding, 0)
    days = (days.join(pol, "full_date", "left")
                .withColumn("is_policy_decision_day", F.col("policy_rate_pct").isNotNull())
                .withColumn("rate_change_bps", F.coalesce("rate_change_bps", F.lit(0)))
                .withColumn("policy_rate_pct", F.last("policy_rate_pct", ignorenulls=True).over(w))
                .where(F.col("full_date") >= F.lit(start)))

    trading = (spark.table("workspace.silver.daily_prices").select(F.col("trade_date").alias("full_date"))
                    .distinct().withColumn("is_trading_day", F.lit(True)))

    dim = (days.join(trading, "full_date", "left")
        .withColumn("is_trading_day", F.coalesce("is_trading_day", F.lit(False)))
        .select(F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), "full_date",
                F.year("full_date").alias("year"), F.quarter("full_date").alias("quarter"),
                F.month("full_date").alias("month"), F.date_format("full_date", "MMMM").alias("month_name"),
                F.date_format("full_date", "yyyy-MM").alias("year_month"),
                F.weekofyear("full_date").alias("week_of_year"), F.date_format("full_date", "EEEE").alias("day_name"),
                F.dayofweek("full_date").isin(1, 7).alias("is_weekend"), "is_trading_day",
                "policy_rate_pct", "rate_change_bps", "is_policy_decision_day",
                F.current_timestamp().alias("load_timestamp")))

    dim.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_date")
    return dim.count()

In [0]:
def equities(df):
    return (df.where(F.col("sector_code").isNotNull())
              .where(~F.col("sector_code").isin("36", "3610"))
              .where(~F.col("symbol").endswith("PS")))

def build_dim_stock():
    s = equities(spark.table("workspace.silver.daily_prices"))

    latest = Window.partitionBy("symbol").orderBy(F.col("trade_date").desc())
    last_row = (s.withColumn("rn", F.row_number().over(latest)).where("rn = 1")
                 .select("symbol", "company_name", "sector_code"))

    stats = s.groupBy("symbol").agg(F.min("trade_date").alias("first_trade_date"),
                                    F.max("trade_date").alias("last_trade_date"),
                                    F.count("*").cast("int").alias("trading_days"))
    max_date = s.agg(F.max("trade_date")).collect()[0][0]
    ref = spark.table("workspace.gold.ref_sector").select("sector_code", "sector_name", "security_type")

    dim = (last_row.join(stats, "symbol").join(ref, "sector_code", "left")
        .select(F.xxhash64("symbol").alias("stock_key"), "symbol",
                F.trim(F.regexp_replace("company_name", r"(XDXB|XDXR|XD|XB|XR)$", "")).alias("company_name"),
                "sector_code",
                F.coalesce("sector_name", F.lit("Legacy sector (pre-2015 codes)")).alias("sector_name"),
                F.coalesce("security_type", F.lit("Legacy")).alias("security_type"),
                "first_trade_date", "last_trade_date", "trading_days",
                (F.col("last_trade_date") >= F.date_sub(F.lit(max_date), 30)).alias("is_active"),
                F.current_timestamp().alias("load_timestamp")))

    dim.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_stock")
    return dim.count()